# Thử lịch sử OHLCV 5 phút từ các nguồn GitHub

Mẫu gồm FPT (HOSE), PVS (HNX) và QNS (UPCoM). Với QNS, mốc 2016 bắt đầu ngày 20/12, phiên đầu tiên của mã ([thông báo HNX](https://www.phs.vn/tin-tuc/hnx-ngay-20122016-ngay-giao-dich-dau-tien-co-phieu-dang-ky-giao-dich-cua-ctcp-duong-quang-ngai-qns/5108965)). Các mốc là mẫu để tìm nguồn có lịch sử cũ. Cần kiểm tra thêm theo từng phiên và từng mã để xác nhận độ phủ 10 năm.

- [vnstock-java-kbs](https://github.com/anhtri04/vnstock-java-kbs/blob/main/docs/KBS_API_DOCUMENTATION.md) dùng KBS `data_5P`.
- [quant-agent](https://github.com/tomtranai/quant-agent/blob/main/quant_agent/technical.py) dùng Entrade Chart. Notebook thử `resolution=5` và kiểm tra timestamp trả về.
- [viet-nam-stock](https://github.com/tran-minh-khoi/viet-nam-stock/blob/main/sources.py) gọi DNSE `/price/ohlc`. Notebook sử dụng Vnstock và key DNSE trong `.env` để gọi cùng API.
- [vnstock-js](https://github.com/ttqteo/vnstock-js/blob/master/src/client/stock.ts) lấy `ONE_MINUTE` từ VCI rồi gộp `5m`, nên bỏ qua trong đợt thử trực tiếp `5m`.

`df` giữ nến theo từng nguồn trong bộ nhớ. Giá giữ nguyên đơn vị nguồn trả về. `coverage` cho biết số dòng hợp lệ và khoảng ngày theo từng truy vấn. Notebook chỉ lưu kết quả thống kê.

In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
import requests
from dotenv import load_dotenv

root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ingest" / "market" / "pipeline.py").is_file()
)
os.chdir(root)
sys.path[:] = [entry for entry in sys.path if Path(entry).resolve() != root / "ingest" / "market"]
sys.path.insert(0, str(root))
load_dotenv(root / ".env", override=True)

from ingest.market.entrade.adapter import ohlcv_5m
from ingest.market.kbs.adapter import probe_minute_history

TIMEZONE = "Asia/Ho_Chi_Minh"
SYMBOLS = {"FPT": "HOSE", "PVS": "HNX", "QNS": "UPCoM"}
WINDOWS = {
    "2016": ("2016-11-15", "2016-11-18"),
    "2021": ("2021-11-15", "2021-11-18"),
    "2026": ("2026-11-16", "2026-11-17"),
}
OHLCV = ["open", "high", "low", "close", "volume"]
COLUMNS = ["time", *OHLCV, "symbol", "source", "interval", "exchange"]


def fetch_kbs(symbol, start, end):
    report, bars = probe_minute_history(symbol, start, end, interval="5m")
    return bars, report["status"], report["http_status"]


def fetch_entrade(symbol, start, end):
    bars = ohlcv_5m(symbol, start, end)
    return bars, "OK" if not bars.empty else "EMPTY", 200


def fetch_dnse(symbol, start, end):
    if not all(os.getenv(key, "").strip() for key in ("DNSE_API_KEY", "DNSE_API_SECRET")):
        return pd.DataFrame(), "MISSING_DNSE_KEYS", None
    from vnstock import Market

    bars = Market("dnse").equity(symbol).ohlcv(start=start, end=end, interval="5m", compat="v5")
    return bars, "OK" if not bars.empty else "EMPTY", None


SOURCES = {"kbs": fetch_kbs, "entrade_chart": fetch_entrade, "dnse": fetch_dnse}

In [2]:
records = []
frames = []
for source, fetch in SOURCES.items():
    for symbol, exchange in SYMBOLS.items():
        for period, (start, end) in WINDOWS.items():
            if symbol == "QNS" and period == "2016":
                start, end = "2016-12-20", "2016-12-23"
            record = dict(source=source, symbol=symbol, exchange=exchange, period=period,
                          start=start, end=end, rows=0, first_time=None, last_time=None,
                          invalid_rows=0, duplicate_rows=0, http_status=None)
            try:
                bars, status, http_status = fetch(symbol, start, end)
                record.update(status=status, http_status=http_status)
                if not bars.empty:
                    times = pd.to_datetime(bars["time"], utc=True, errors="coerce").dt.tz_convert(TIMEZONE)
                    lower = pd.Timestamp(start, tz=TIMEZONE)
                    upper = pd.Timestamp(end, tz=TIMEZONE) + pd.Timedelta(days=1)
                    numeric = bars[OHLCV].apply(pd.to_numeric, errors="coerce").replace([float("inf"), -float("inf")], float("nan"))
                    valid = (times.between(lower, upper, inclusive="left")
                             & times.eq(times.dt.floor("5min"))
                             & numeric.notna().all(axis=1)
                             & numeric["open"].between(numeric["low"], numeric["high"])
                             & numeric["close"].between(numeric["low"], numeric["high"])
                             & numeric["low"].gt(0) & numeric["volume"].ge(0))
                    checked = numeric.loc[valid].assign(time=times.loc[valid])
                    unique = checked.drop_duplicates("time")
                    record.update(status="OK" if not unique.empty else "NO_VALID_BARS",
                                  rows=len(unique), first_time=unique["time"].min(),
                                  last_time=unique["time"].max(), invalid_rows=int((~valid).sum()),
                                  duplicate_rows=len(checked) - len(unique))
                    if not unique.empty:
                        frames.append(unique.assign(symbol=symbol, source=source,
                                                    interval="5m", exchange=exchange)[COLUMNS])
            except requests.HTTPError as exc:
                record.update(status="HTTP_ERROR", http_status=exc.response.status_code)
            except Exception as exc:
                record.update(status=type(exc).__name__)
            records.append(record)

coverage = pd.DataFrame(records).sort_values(["source", "period", "symbol"])
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=COLUMNS)
print(coverage[["source", "period", "symbol", "status", "http_status", "rows", "invalid_rows",
                "duplicate_rows", "first_time", "last_time"]].to_string(index=False))

       source period symbol  status  http_status  rows  invalid_rows  duplicate_rows                first_time                 last_time
         dnse   2016    FPT   EMPTY          NaN     0             0               0                       NaT                       NaT
         dnse   2016    PVS   EMPTY          NaN     0             0               0                       NaT                       NaT
         dnse   2016    QNS   EMPTY          NaN     0             0               0                       NaT                       NaT
         dnse   2021    FPT   EMPTY          NaN     0             0               0                       NaT                       NaT
         dnse   2021    PVS   EMPTY          NaN     0             0               0                       NaT                       NaT
         dnse   2021    QNS   EMPTY          NaN     0             0               0                       NaT                       NaT
         dnse   2026    FPT      OK      

In [3]:
by_source = coverage.groupby(["source", "period"])["rows"].sum().unstack(fill_value=0)
print(by_source.to_string())
print()
print(f"Số dòng theo nguồn: {len(df):,}")
print(f"Số cặp mã và timestamp khác nhau: {len(df.drop_duplicates(['symbol', 'time'])):,}")

period         2016  2021  2026
source                         
dnse              0     0   252
entrade_chart     0     0   252
kbs               0     0     0

Số dòng theo nguồn: 504
Số cặp mã và timestamp khác nhau: 252


In [4]:
df.head(10)